In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Жесткая фиксация среды выполнения для Spark
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Указываем путь к нашей новой папке
os.environ['HADOOP_HOME'] = r'C:\hadoop'
# Добавляем её в системный путь, чтобы Spark нашел файлы
os.environ['PATH'] = os.environ['HADOOP_HOME'] + r'\bin;' + os.environ.get('PATH', '')

def create_spark_session():
    print("--> Инициализация ядра PySpark...")
    return SparkSession.builder \
        .appName("NYTaxiCleaner") \
        .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
        .getOrCreate()

In [2]:
spark = create_spark_session()
spark.version

--> Инициализация ядра PySpark...


'4.1.1'

In [127]:
spark.stop()

In [3]:
from pyspark.sql.functions import col
from pathlib import Path

In [4]:
#BASE_DIR = Path(__file__).resolve().parent.parent
ZONES_FLAGS = 'C:/Users/Fox/repository/ny_taxi_pipeline/data/raw/taxi+_zone_lookup.csv'
PROCESSED_DATA_PATH = r"C:\Users\Fox\repository\ny_taxi_pipeline\data\processed\cleaned_yellow_tripdata"

In [5]:
zones_schema = StructType([
    StructField("LocationID", IntegerType(), True),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True)
])

In [6]:
df_fact = spark.read.parquet(PROCESSED_DATA_PATH)
df_zone = spark.read.csv(ZONES_FLAGS, schema=zones_schema)

In [7]:
df_fact.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [ ]:
df_fact.select('total_amount').show(3)

+------------+
|total_amount|
+------------+
|        22.7|
|       18.75|
|        31.3|
+------------+
only showing top 3 rows


In [9]:
full_df =  df_fact.join(
        F.broadcast(df_zone).alias('pu'), on=(F.col('PULocationID') == F.col('pu.LocationID'))
        , how='left').join(
        F.broadcast(df_zone).alias('do'), on=(F.col('DOLocationID') == F.col('do.LocationID'))
        , how='left').select(
            [
                F.hour('tpep_pickup_datetime').alias('hour_of_day')
            ] +
            df_fact.columns + [
                F.col('pu.Borough').alias('PU_borough'),
                F.col('pu.Zone').alias('PU_Zone'),
                F.col('pu.service_zone').alias('PU_service_zone'),
                F.col('do.Borough').alias('DO_borough'),
                F.col('do.Zone').alias('DO_Zone'),
                F.col('do.service_zone').alias('DO_service_zone')
            ]
        ).withColumn(
            'part_of_day',
            F.when((F.hour('tpep_pickup_datetime') >= 0) & (F.hour('tpep_pickup_datetime') < 6), '1_night').\
            when((F.hour('tpep_pickup_datetime') >= 6) & (F.hour('tpep_pickup_datetime') < 12), '2_morning').\
            when((F.hour('tpep_pickup_datetime') >= 12) & (F.hour('tpep_pickup_datetime') < 18), '3_day').\
            otherwise('4_evening')
        ).groupBy('part_of_day', 'PU_borough', 'DO_borough').agg(
            F.round(F.sum("total_amount"), 2).alias('total_amount'),
            F.round(F.avg('total_amount'), 2).alias('avg_amount'),
            F.round(F.sum('trip_distance'), 2).alias('sum_trip_distance'),
            F.round(F.avg('trip_distance'), 2).alias('avg_trip_distance')
        ).withColumn(
            'order',
            F.row_number().over(Window.partitionBy('part_of_day').orderBy(F.col('total_amount').desc()))
            ).filter(F.col('order') <= 5).drop('order').\
                orderBy(F.col('part_of_day'), F.col('total_amount').desc())

In [10]:
full_df.show()

+-----------+----------+----------+-------------+----------+-----------------+-----------------+
|part_of_day|PU_borough|DO_borough| total_amount|avg_amount|sum_trip_distance|avg_trip_distance|
+-----------+----------+----------+-------------+----------+-----------------+-----------------+
|    1_night| Manhattan| Manhattan|   3519585.37|     20.72|        372773.74|             2.19|
|    1_night|    Queens| Manhattan|    609862.72|     80.66|        115526.88|            15.28|
|    1_night| Manhattan|    Queens|    554928.43|     53.73|          95864.7|             9.28|
|    1_night| Manhattan|  Brooklyn|    379940.38|     38.89|         57072.23|             5.84|
|    1_night|    Queens|  Brooklyn|    252055.62|     67.34|         52647.07|            14.07|
|  2_morning| Manhattan| Manhattan|1.071112323E7|     19.62|        995880.96|             1.82|
|  2_morning|    Queens| Manhattan|   2252268.71|     78.22|        384995.36|            13.37|
|  2_morning| Manhattan|    Qu

In [ ]:
full_df.show(3)
#full_df.select(hour('tpep_pickup_datetime')).show(3)

+-----------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+--------------------+---------------+----------+--------------------+---------------+
|hour_of_day|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|PU_borough|             PU_Zone|PU_service_zone|DO_borough|             DO_Zone|DO_service_zone|
+-----------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------

In [ ]:
df_zone.columns

['LocationID', 'Borough', 'Zone', 'service_zone']